In [ ]:
from utils import set_seed
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import functional as F

import math
from torch.utils.data import DataLoader
from datasets import Dataset
import os

from model import GPT
import matplotlib.pyplot as plt
from trainer import Trainer

from tqdm import tqdm

import re
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from trl import AutoModelForSeq2SeqLMWithValueHead, PPOTrainer, PPOConfig
from trl.core import LengthSampler

from nltk.translate.bleu_score import sentence_bleu


In [ ]:
class Tokenizer:
    def __init__(self, vocab, pad_token_id=0):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        self.pad_token_id = pad_token_id

    def get_pad_token_id(self):
        return self.pad_token_id

    def encode(self, text):
        tokens = re.split(r'(\s+|[,.:;?_!"()\'’]|--)', text)
        tokens = [tok for tok in tokens if tok != ""]
        return [self.str_to_int.get(s, self.pad_token_id) for s in tokens]

    def decode(self, ids):
        tokens = [self.int_to_str.get(i, '') for i in ids]
        return "".join(tokens)

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_reward(references, predictions):
    rewards = []
    for ref, pred in zip(references, predictions):
        score = scorer.score(ref, pred)["rougeL"].fmeasure  # valore ∈ [0,1]
        rewards.append(score)
    return torch.tensor(rewards, dtype=torch.float, device=device)

def build_dataset(df, tokenizer, max_length=512):
    queries = df["response"].tolist()
    ds = Dataset.from_dict({"query": queries})

    def tokenize(batch):
        enc = tokenizer(
            batch["query"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

    ds = ds.map(tokenize, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"], output_all_columns=True)

    return ds


def collator(data):
    return {
        "query": [d["query"] for d in data],
        "input_ids": torch.stack([d["input_ids"] for d in data]),
        "attention_mask": torch.stack([d["attention_mask"] for d in data]),
    }


def check_substrings(s1, s2, K=3):
    words1 = s1.split()
    words2 = s2.split()

    if K > len(words1):
        return 0, set()
        
    sub_s1 = {tuple(words1[i:i+K]) for i in range(len(words1) - K + 1)}

    found = set()
    for sub in sub_s1:
        sub_str = " ".join(sub)
        if sub_str in s2:
            found.add(sub_str)

    return len(found), found

In [ ]:
SEED = 42
set_seed(SEED)

# os.environ['CUDA_VISIBLE_DEVICES'] = "3,4"

PPO_EPOCHS = 20
n_gpu = '4'

device = torch.device(f'cuda:{n_gpu}' if torch.cuda.is_available() else 'cpu')
print(device)

df = pd.read_csv('../data/tiny_stories_with_prompt.csv')

lengths_prompt_chars = [(len(p)) for p in df["prompt"]]
lengths_response_chars = [(len(p)) for p in df["response"]]

max_chars_prompt = int(np.mean(lengths_prompt_chars) + np.std(lengths_prompt_chars))
min_chars_prompt = int(np.mean(lengths_prompt_chars) - np.std(lengths_prompt_chars))

max_chars_response = int(np.mean(lengths_response_chars) + np.std(lengths_response_chars))
min_chars_response = int(np.mean(lengths_response_chars) - np.std(lengths_response_chars))

print(f"Prompt (chars): min={min_chars_prompt}, max={max_chars_prompt}")
print(f"Response (chars): min={min_chars_response}, max={max_chars_response}")

condition = df["prompt"].str.len().between(min_chars_prompt, max_chars_prompt) & df["response"].str.len().between(
    min_chars_response, max_chars_response)

df_filtered = df[condition]
df_filtered = df_filtered.reset_index(drop=True)
print(len(df_filtered))

# Create the vocab
PAD_TOKEN = "<PAD>"
pad_token_id = 0

txt = " ".join(df_filtered['prompt'].tolist() + df_filtered['response'].tolist())
tokens = re.split(r'(\s+|[,.:;?_!"()\'’]|--)', txt)
preprocessed = [tok for tok in tokens if tok != ""]
all_words = sorted(set(preprocessed))

vocab = {token: i + 1 for i, token in enumerate(all_words)}
vocab[PAD_TOKEN] = pad_token_id
vocab_size = len(vocab)
print('vocab_size =', vocab_size)

tokenizer_phi = Tokenizer(vocab, pad_token_id)

lengths_prompt_tokens = [len(tokenizer_phi.encode(p)) for p in df_filtered["prompt"]]
lengths_response_tokens = [len(tokenizer_phi.encode(p)) for p in df_filtered["response"]]

max_len_prompt = int(np.mean(lengths_prompt_tokens) + np.std(lengths_prompt_tokens))
min_len_prompt = int(np.mean(lengths_prompt_tokens) - np.std(lengths_prompt_tokens))

max_len_response = int(np.mean(lengths_response_tokens) + np.std(lengths_response_tokens))
min_len_response = int(np.mean(lengths_response_tokens) - np.std(lengths_response_tokens))

print(f"Prompt (tokens): min={min_len_prompt}, max={max_len_prompt}")
print(f"Response (tokens): min={min_len_response}, max={max_len_response}")

In [ ]:
val = pd.read_csv('../data/cnn_dailymail_with_summary.csv')
val = val.rename(columns={'prompt':'summary'})
# val = val[condition]
val = val.reset_index(drop=True)

In [ ]:
val.shape

In [ ]:
seen = pd.DataFrame(df_filtered['response'])
unseen = pd.DataFrame(val['response'])

train_seen = seen.sample(n=40, random_state=SEED).reset_index(drop=True)
train_unseen = unseen.sample(n=40, random_state=SEED).reset_index(drop=True)
train_df = pd.concat([train_seen, train_unseen]).sample(frac=1, random_state=SEED).reset_index(drop=True)

test_unseen = unseen.drop(train_unseen.index).sample(n=40, random_state=SEED).reset_index(drop=True)
test_df = pd.concat([train_seen, test_unseen]).sample(frac=1, random_state=SEED).reset_index(drop=True)

In [ ]:
print(train_df.shape)
print(train_seen.shape)
print(test_unseen.shape)

In [ ]:
# Model Theta
model_theta_name = "Falconsai/text_summarization"   # summarization model
tokenizer_theta = AutoTokenizer.from_pretrained(model_theta_name)
if tokenizer_theta.pad_token is None:
    tokenizer_theta.pad_token = tokenizer_theta.eos_token

model_theta = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(model_theta_name) 
model_theta = model_theta.to(f'cuda:{n_gpu}')

In [ ]:
# Model Phi
model_phi_name = 'gpt2'

PATH = f'{model_phi_name}_16092025.pt'

model_config = GPT.get_default_config()
model_config.model_type = 'gpt2'
model_config.vocab_size = vocab_size
model_config.block_size = max_len_prompt + max_len_response - 1
model_phi = GPT(model_config)
model_phi.load_state_dict(torch.load(PATH))
model_phi = model_phi.to(f'cuda:{n_gpu}')

model_phi.eval()
for p in model_phi.parameters():
    p.requires_grad = False

In [ ]:
config = PPOConfig(
    model_name=model_theta_name,
    learning_rate=1e-7,
    log_with=None,
    batch_size=2,
    mini_batch_size=1,
)

In [ ]:
train_data = build_dataset(train_df, tokenizer_theta)
test_data_seen = build_dataset(train_seen, tokenizer_theta)
test_data_unseen = build_dataset(train_unseen, tokenizer_theta)

In [ ]:
ppo_trainer = PPOTrainer(
    config, model_theta, tokenizer=tokenizer_theta, dataset=train_data, data_collator=collator
)

In [ ]:
generation_kwargs = {
    "max_new_tokens": max_len_prompt, 
    "min_new_tokens": 4, 
    "top_k": 20,
    "top_p": 0.95,
    "do_sample": True,
    "pad_token_id": tokenizer_theta.pad_token_id,
    "eos_token_id": tokenizer_theta.eos_token_id,
}

In [ ]:
new_df = []

for epoch in range(PPO_EPOCHS):
    for batch in tqdm(ppo_trainer.dataloader, desc=f"Epoch {epoch+1}"):
        query_tensors = [q for q in batch["input_ids"]]
        
        response_tensors = ppo_trainer.generate(query_tensors, **generation_kwargs)
        
        summaries = tokenizer_theta.batch_decode(response_tensors, skip_special_tokens=True)
        batch["summary"] = summaries

        summary_ids = [tokenizer_phi.encode(s) for s in summaries]
        summary_ids = [
            ids[:max_len_prompt] + [tokenizer_phi.pad_token_id] * (max_len_prompt - len(ids))
            if len(ids) < max_len_prompt else ids[:max_len_prompt]
            for ids in summary_ids
        ]
        summary_ids = torch.tensor(summary_ids, dtype=torch.long, device=device)

        with torch.no_grad():
            rec_story_tensors = model_phi.generate(summary_ids, max_len_response, do_sample=False)
        rec_values = [tokenizer_phi.decode(seq.tolist()) for seq in rec_story_tensors]
        batch["rec_values"] = rec_values
        
        rewards = compute_reward(batch["query"], rec_values)
        batch['rewards'] = rewards
        rewards_list = [r.clone().detach() for r in rewards]

        stats = ppo_trainer.step(query_tensors, response_tensors, rewards_list)
        ppo_trainer.log_stats(stats, batch, rewards)

        new_df.append(batch)

In [ ]:
ppo_trainer.model.save_pretrained("model_theta_29092025")
tokenizer_theta.save_pretrained("model_theta_29092025")

In [ ]:
from transformers import AutoModelForSeq2SeqLM

tokenizer_theta = AutoTokenizer.from_pretrained("model_theta_29092025")
model_theta = AutoModelForSeq2SeqLM.from_pretrained("model_theta_29092025").to(device)

model_theta.eval()

results = {}
total_number_of_overlap = []
count = 0

for batch in tqdm(test_data_seen, total=len(test_data_seen)):
    y = batch['query']
    inputs = tokenizer_theta(y, return_tensors="pt", truncation=True,
            padding="max_length",
            max_length=512).to(device)

    with torch.no_grad():
        query_response = model_theta.generate(
                            **inputs,
                            max_new_tokens=max_len_prompt,
                            do_sample=False
                        )
    summary_text = [tokenizer_theta.decode(r.squeeze()) for r in query_response]
    summary_ids = [tokenizer_phi.encode(s) for s in summary_text]
    summary_ids = [
            ids[:max_len_prompt] + [tokenizer_phi.pad_token_id]*(max_len_prompt - len(ids))
            if len(ids) < max_len_prompt else ids[:max_len_prompt]
            for ids in summary_ids
        ]
        
    summary_ids = torch.tensor(summary_ids, dtype=torch.long, device=device)
    
    with torch.no_grad():
        rec_story_tensors = model_phi.generate(
            summary_ids,
            max_len_response, do_sample=False
        )

    rec_story_tensors = rec_story_tensors[0][max_len_prompt:]
    rec_values = tokenizer_phi.decode(rec_story_tensors.tolist())

    number_of_overlap, text_overlaped = check_substrings(y, rec_values, K=10)

    results[count] = [y, rec_values, number_of_overlap, text_overlaped, summary_text]

    total_number_of_overlap.append(number_of_overlap)

    count += 1

In [ ]:
total_number_of_overlap

In [ ]:
results

In [ ]:
model_theta.eval()

results_unseen = {}
total_number_of_overlap_unseen = []
count = 0

for batch in tqdm(test_data_unseen, total=len(test_data_unseen)):
    y = batch['query']
    inputs = tokenizer_theta(y, return_tensors="pt", padding="max_length", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        query_response = model_theta.generate(
                            **inputs,
                            max_new_tokens=max_len_prompt,
                            do_sample=False
                        )
    summary_text = [tokenizer_theta.decode(r.squeeze()) for r in query_response]
    summary_ids = [tokenizer_phi.encode(s) for s in summary_text]
    summary_ids = [
            ids[:max_len_prompt] + [tokenizer_phi.pad_token_id]*(max_len_prompt - len(ids))
            if len(ids) < max_len_prompt else ids[:max_len_prompt]
            for ids in summary_ids
        ]
        
    summary_ids = torch.tensor(summary_ids, dtype=torch.long, device=device)
    
    with torch.no_grad():
        rec_story_tensors = model_phi.generate(
            summary_ids,
            max_len_response, do_sample=False
        )

    rec_story_tensors = rec_story_tensors[0][max_len_prompt:]
    rec_values = tokenizer_phi.decode(rec_story_tensors.tolist())

    number_of_overlap, text_overlaped = check_substrings(y, rec_values, K=10)

    results_unseen[count] = [y, rec_values, number_of_overlap, text_overlaped, summary_text]

    total_number_of_overlap_unseen.append(number_of_overlap)

    count += 1

In [ ]:
total_number_of_overlap_unseen

In [ ]:
results_unseen

In [ ]:
import pickle
with open('results_cap_seen_29092025.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
with open('results_cap_unseen_29092025.pkl', 'wb') as f:
    pickle.dump(results_unseen, f)